# DRiLL Khoản–Điểm verification suite (Kaggle T4 x2)

Run all cells in order. Completed cases are skipped on a same-session rerun. Every experiment saves fixed top-1..5 metrics; reranker runs also tune score thresholds with top-1/top-2 fallback on the persisted dev split. The final cases are ordered so useful artifacts survive if the session ends before the optional structure control.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import os
import subprocess

secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Kaggle Secret HF_TOKEN is missing or empty')
os.environ['HF_TOKEN'] = hf_token

token = secrets.get_secret('GITHUB_TOKEN')
repo_url = 'github.com/leeminsun1205/Drill-reliable-retrieval.git'
exit_code = os.system(f'git clone https://oauth2:{token}@{repo_url}')
if exit_code != 0:
    raise RuntimeError('Git clone failed')

%cd /kaggle/working/Drill-reliable-retrieval

In [ ]:
!python -m pip install -e '.[dense,vietnamese]'

!python scripts/validate_data.py \
  --corpus /kaggle/input/datasets/lmnhnht125/wedev-vlsp/legal-corpus.parquet \
  --queries /kaggle/input/datasets/lmnhnht125/wedev-vlsp/train.parquet

vncorenlp_dir = Path('/kaggle/working/VnCoreNLP')
if not vncorenlp_dir.exists():
    !git clone --depth 1 https://github.com/vncorenlp/VnCoreNLP.git /kaggle/working/VnCoreNLP
!ls -lh /kaggle/working/VnCoreNLP/VnCoreNLP-1.2.jar
!ls -lh /kaggle/working/VnCoreNLP/models/wordsegmenter

## Environment preflight — fail fast before the overnight run

Kiểm tra toàn bộ điều kiện dễ gây lỗi muộn: đúng hai GPU, Java và VnCoreNLP assets, HF token hợp lệ, truy cập được tokenizer của cả dense model lẫn reranker, và dung lượng ghi còn đủ. Cell này không chạy retrieval.

In [ ]:
import shutil
import torch
from huggingface_hub import HfApi
from transformers import AutoTokenizer
import pyarrow.parquet as pq

gpu_count = torch.cuda.device_count()
if gpu_count != 2:
    raise RuntimeError(f'Expected Kaggle T4 x2, found {gpu_count} CUDA device(s)')
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(gpu_count)])

jar_path = Path('/kaggle/working/VnCoreNLP/VnCoreNLP-1.2.jar')
segmenter_dir = Path('/kaggle/working/VnCoreNLP/models/wordsegmenter')
if not jar_path.is_file() or not segmenter_dir.is_dir():
    raise RuntimeError('VnCoreNLP JAR or word-segmentation models are missing')
java_check = subprocess.run(['java', '-version'], capture_output=True, text=True)
if java_check.returncode != 0:
    raise RuntimeError(f'Java is unavailable: {java_check.stderr}')
print(java_check.stderr.splitlines()[0])

identity = HfApi().whoami(token=os.environ['HF_TOKEN'])
print('Hugging Face authentication:', identity.get('name', 'authenticated'))
loaded_tokenizers = {}
for model_name in ('BAAI/bge-m3', 'BAAI/bge-reranker-v2-m3'):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if not getattr(tokenizer, 'is_fast', False):
        raise RuntimeError(f'{model_name} did not provide a fast tokenizer')
    loaded_tokenizers[model_name] = tokenizer
    print(f'Tokenizer ready: {model_name}')

reranker_tokenizer = loaded_tokenizers['BAAI/bge-reranker-v2-m3']
query_questions = pq.read_table(
    '/kaggle/input/datasets/lmnhnht125/wedev-vlsp/train.parquet',
    columns=['question'],
).column('question').to_pylist()
query_content_lengths = [
    len(reranker_tokenizer(str(question), add_special_tokens=False)['input_ids'])
    for question in query_questions
]
max_query_content = max(query_content_lengths)
pair_special_tokens = int(reranker_tokenizer.num_special_tokens_to_add(pair=True))
document_special_tokens = int(reranker_tokenizer.num_special_tokens_to_add(pair=False))
safe_document_max = 512 - max_query_content - pair_special_tokens + document_special_tokens
if safe_document_max < 448:
    raise RuntimeError(
        f'Reranker document budget 448 is unsafe; observed safe maximum is {safe_document_max}'
    )
print(
    'Reranker-512 budget:',
    {
        'max_query_content': max_query_content,
        'pair_special_tokens': pair_special_tokens,
        'safe_document_max': safe_document_max,
        'configured_document_max': 448,
    },
)

free_gib = shutil.disk_usage('/kaggle/working').free / (1024 ** 3)
if free_gib < 8:
    raise RuntimeError(f'Only {free_gib:.1f} GiB free under /kaggle/working')
print(f'Free /kaggle/working space: {free_gib:.1f} GiB')
print('Environment preflight passed')

In [ ]:
import json
import pandas as pd
import time
from IPython.display import display

RESULTS = Path('/kaggle/working/results')
CORPUS = '/kaggle/input/datasets/lmnhnht125/wedev-vlsp/legal-corpus.parquet'
TRAIN = '/kaggle/input/datasets/lmnhnht125/wedev-vlsp/train.parquet'
JAR = '/kaggle/working/VnCoreNLP/VnCoreNLP-1.2.jar'
SUITE_STARTED = time.monotonic()
OPTIONAL_START_DEADLINE_HOURS = 6.0

def completed(name):
    return (RESULTS / name / 'metrics_by_output_k.json').exists()

def show_case(name):
    run_dir = RESULTS / name
    fixed_path = run_dir / 'metrics_by_output_k.json'
    if not fixed_path.exists():
        print(f'INCOMPLETE: {name}')
        return
    fixed = json.loads(fixed_path.read_text(encoding='utf-8'))['results']
    rows = [{'output_k': int(k), **v} for k, v in fixed.items()]
    print(f'\n{name} — fixed output-k')
    display(pd.DataFrame(rows)[['output_k', 'precision', 'recall', 'f2', 'queries']])
    threshold_path = run_dir / 'metrics_by_score_threshold.json'
    if threshold_path.exists():
        payload = json.loads(threshold_path.read_text(encoding='utf-8'))
        threshold_rows = []
        for fallback, values in payload['best_by_fallback_k'].items():
            threshold_rows.append({
                'fallback_k': int(fallback),
                'threshold': values['threshold'],
                'mean_output': values['output_size']['mean'],
                'precision': values['precision'],
                'recall': values['recall'],
                'f2': values['f2'],
            })
        print(f'{name} — dev-tuned threshold (diagnostic)')
        display(pd.DataFrame(threshold_rows))

## 1. Strong BM25 control — VnCoreNLP

**Mục tiêu:** tái lập sparse baseline mạnh nhất đã đo, dùng tách từ tiếng Việt thay cho regex mặc định.  
**Cấu hình chính:** BM25 `k1=1.2`, `b=0.75`, VnCoreNLP, candidate top 100.  
**Vai trò:** lexical control; không dùng GPU và không có dense/reranker.  
**Cách đọc:** so F2 top 1–5 để thấy riêng BM25 phù hợp trả bao nhiêu article; Recall@K cho biết trần candidate của nhánh sparse.

In [ ]:
CASE = 'bm25_vncorenlp_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/bm25.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --output-root {RESULTS}
show_case(CASE)

## 2. Non-structural dense control — BGE-M3 token windows 512/64

**Mục tiêu:** tạo dense control không dùng cấu trúc pháp luật để so công bằng với Khoản–Điểm ở case 3.  
**Cấu hình chính:** `BAAI/bge-m3`, fixed tokenizer window 512 token, overlap 64, max aggregation từ chunk về `aid`.  
**Biến được kiểm soát:** cùng model, split, token budget, overlap, search và output policy với case 3; chỉ boundary strategy khác.  
**Cách đọc:** case 3 chỉ chứng minh lợi ích của cấu trúc Khoản–Điểm nếu vượt case này, không chỉ vượt article truncation. Embeddings được cache để các case sau tái sử dụng.

In [ ]:
CASE = 'dense_bge_m3_token_window512_64_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/dense.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --model-name BAAI/bge-m3 \
      --query-prefix "" \
      --document-prefix "" \
      --max-seq-length 512 \
      --encode-batch-size 4 \
      --devices 2 \
      --set retrieval.dense.chunking.method=token_window \
      --set retrieval.dense.chunking.max_tokens=512 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --output-root {RESULTS}
show_case(CASE)

## 3. Hierarchical structure treatment — BGE-M3 Khoản–Điểm 512/64

**Mục tiêu:** giữ nguyên Khoản nếu vừa 512; Khoản quá dài mới greedy-pack các Điểm và window Điểm cá biệt quá dài.  
**Cấu hình chính:** giữ nguyên BGE-M3 và ngân sách 512/64 của case 2, đổi duy nhất representation sang `khoan_diem`.  
**Vai trò:** structure-aware treatment và là dense branch đầu vào cho hybrid/reranker phía sau.  
**Cách đọc:** so trực tiếp top-k final và Recall@K với case 2; cache của run này phải được hit ở case 4–7.

In [ ]:
CASE = 'dense_bge_m3_khoan_diem512_64_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/dense.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --model-name BAAI/bge-m3 \
      --query-prefix "" \
      --document-prefix "" \
      --max-seq-length 512 \
      --encode-batch-size 4 \
      --devices 2 \
      --set retrieval.dense.chunking.method=khoan_diem \
      --set retrieval.dense.chunking.max_tokens=512 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --output-root {RESULTS}
show_case(CASE)

## 4. Khoản–Điểm hybrid first stage — VnCoreNLP + BGE-M3 + WRRF20

**Mục tiêu:** xác nhận first-stage mạnh dùng chung cho mọi reranker, không thay candidate giữa các ablation.  
**Cấu hình chính:** BM25 VnCoreNLP + dense Khoản–Điểm BGE-M3, weighted RRF `k=20`, sparse/dense `0.2/0.8`, mỗi nhánh top 100.  
**Vai trò:** candidate generator; chưa có cross-encoder. Dense embeddings từ case 3 được tái sử dụng nên run này chủ yếu tốn BM25 và fusion.  
**Cách đọc:** Recall@50 là trần candidate cho reranker pool 50; fixed top 1–5 cho biết fusion tự thân mạnh tới đâu.

In [ ]:
CASE = 'hybrid_vncorenlp_bge_m3_khoan_diem_wrrf20_d08_s02_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --model-name BAAI/bge-m3 \
      --query-prefix "" \
      --document-prefix "" \
      --max-seq-length 512 \
      --encode-batch-size 4 \
      --devices 2 \
      --set retrieval.dense.chunking.method=khoan_diem \
      --set retrieval.dense.chunking.max_tokens=512 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set retrieval.hybrid.sparse_weight=0.2 \
      --set retrieval.hybrid.dense_weight=0.8 \
      --set retrieval.hybrid.rrf_k=20 \
      --output-root {RESULTS}
show_case(CASE)

## 5. Khoản–Điểm reranker — pair 1024 / document 768

**Mục tiêu:** tái lập zero-shot reranker mạnh hiện tại và bổ sung selection-policy analysis mà không chạy model nhiều lần.  
**Cấu hình chính:** frozen hybrid case 4, rerank top 50 bằng `BAAI/bge-reranker-v2-m3`, Khoản–Điểm representation, document 768 và pair max 1024.  
**Biến giữ cố định:** first-stage candidates, split, model và chunk aggregation; đây là reference cho case 6.  
**Cách đọc:** ngoài fixed top 1–5, bảng threshold quét chính xác logits trên dev với fallback 1/2. Threshold là dev-tuned diagnostic, chưa phải test estimate unbiased.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem_wrrf20_reranker1024_768_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set reranking.representation=khoan_diem \
      --set reranking.document_max_tokens=768 \
      --set reranking.max_length=1024 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
show_case(CASE)

## 6. Reranker document-budget control — same dense Khoản–Điểm 512, pair 1024

**Mục tiêu:** kiểm tra reranker có cần giữ nguyên Khoản dài tới 768 token hay trần cấu trúc 512 đã đủ.  
**Cấu hình chính:** giữ nguyên dense Khoản–Điểm 512/64, hybrid candidates, reranker model và pair max 1024 của case 5; chỉ hạ reranker structure budget `768 → 512`.  
**Hành vi:** Khoản ≤512 được giữ nguyên; Khoản dài mới xuống greedy Điểm packs hoặc token-window fallback.  
**Cách đọc:** first-stage Recall@K phải giống case 5. Chênh lệch final F2/runtime/representation count khi đó được quy cho reranker document fallback 768 so với 512.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem512_wrrf20_reranker1024_512_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set reranking.representation=khoan_diem \
      --set reranking.document_max_tokens=512 \
      --set reranking.max_length=1024 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
show_case(CASE)

## 7. End-to-end efficiency treatment — Khoản–Điểm 448 + reranker pair 512

**Mục tiêu:** đo chất lượng khi hạ trần Khoản–Điểm xuống 448 để tối ưu end-to-end cho pair 512.  
**Cấu hình chính:** dense và reranker đều dùng `khoan_diem`; Khoản ≤448 được giữ nguyên, Khoản dài mới xuống Điểm packs/window. Reranker đổi pair `1024 → 512`.  
**Lý do 448:** dành xấp xỉ 64 token cho query và pair overhead; `truncation=only_second` vẫn bảo vệ toàn bộ query.  
**Cách đọc:** đây là end-to-end efficiency treatment, không phải reranker-only ablation. So first-stage Recall@K, final F2, chunking stats, representation count và runtime với case 5.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem448_wrrf20_reranker_khoan_diem512_448_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set retrieval.dense.chunking.max_tokens=448 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --set reranking.representation=khoan_diem \
      --set reranking.document_max_tokens=448 \
      --set reranking.max_length=512 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
show_case(CASE)

## 8. Direct evidence reranking — top 50 retrieved chunks, then max to article

**Mục tiêu:** kiểm tra đề xuất rerank đúng các evidence chunks đã được retrieval chọn, thay vì lấy top-50 article rồi mở lại mọi chunk của chúng.  
**Cấu hình chính:** dense lấy top 100 Khoản–Điểm chunks; BM25 article rank và dense chunk rank được fusion bằng WRRF20; chọn đúng 50 chunks để cross-encoder chấm, sau đó mới max aggregation theo `aid`. Pair max vẫn là 1024 nên đây không phải length ablation.  
**Chi phí:** đúng 50 cặp/query; một `aid` có thể chiếm nhiều chunk nên output candidate có thể ít hơn 50 article. `reranker_chunk_candidates.jsonl` cho phép đo duplication/coverage.  
**Cách đọc:** so F2, runtime và Recall@K candidate pool với case 5. Case này thay đổi candidate unit, không phải bản tối ưu tương đương toán học của article-level hybrid.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem_wrrf20_rerank_retrieved_chunks50_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set reranking.candidate_source=retrieved_chunks \
      --set reranking.retrieved_chunk_depth=100 \
      --set reranking.candidate_pool_size=50 \
      --set reranking.max_length=1024 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
show_case(CASE)

## 8b. Efficient direct evidence — Khoản–Điểm 448/64, top 50 chunks, pair 512

**Mục tiêu:** kết hợp candidate flow chỉ chấm 50 chunks của case 8 với budget hiệu quả 448/64 và pair 512 của case 7.  
**So sánh chính:** case 7 giữ cùng dense representation, WRRF và pair budget nhưng aggregate thành top-50 articles trước khi mở lại mọi representation. Case này chọn top-50 fused chunks trước reranker.  
**Chi phí kỳ vọng:** đúng 50 query–chunk pairs mỗi query. `reranking.document_max_tokens` không tham gia vì retrieved dense chunks đã thỏa budget 448.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem448_wrrf20_rerank_retrieved_chunks50_512_seed42_multik'
if not completed(CASE):
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set retrieval.dense.chunking.max_tokens=448 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --set reranking.candidate_source=retrieved_chunks \
      --set reranking.retrieved_chunk_depth=100 \
      --set reranking.candidate_pool_size=50 \
      --set reranking.max_length=512 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
show_case(CASE)

## 9. Optional reranker-structure control — dense Khoản–Điểm 448 + token-window reranker

**Mục tiêu:** cô lập lợi ích của cấu trúc pháp luật ngay tại tầng reranker.  
**Cấu hình chính:** tái sử dụng dense Khoản–Điểm 448/64 và first-stage candidates của case 7; giữ reranker model cùng pair/document budget 512/448, chỉ đổi reranker representation `khoan_diem → token_window`.  
**Vai trò:** non-structural reranker control; chạy cuối vì đắt và ít ưu tiên hơn các mốc chính. Chỉ bắt đầu nếu suite chưa vượt 6 giờ.  
**Cách đọc:** nếu case 7 vượt case 9 thì Khoản–Điểm giúp reranker dưới cùng ngân sách; nếu không, lợi ích cấu trúc chủ yếu có thể nằm ở first stage.

In [ ]:
CASE = 'hybrid_bge_m3_khoan_diem448_wrrf20_reranker_token_window512_448_seed42_multik'
elapsed_hours = (time.monotonic() - SUITE_STARTED) / 3600
if not completed(CASE) and elapsed_hours < OPTIONAL_START_DEADLINE_HOURS:
    !python scripts/run_experiment.py \
      --config configs/hybrid_reranker_khoan_diem.yaml \
      --corpus-path {CORPUS} \
      --train-path {TRAIN} \
      --experiment-name {CASE} \
      --devices 2 \
      --set retrieval.bm25.preprocessing.tokenizer.word_segmenter=vncorenlp \
      --set retrieval.bm25.preprocessing.tokenizer.vncorenlp_jar_path={JAR} \
      --set retrieval.dense.chunking.max_tokens=448 \
      --set retrieval.dense.chunking.overlap_tokens=64 \
      --set reranking.representation=token_window \
      --set reranking.document_max_tokens=448 \
      --set reranking.max_length=512 \
      --set reranking.batch_size=8 \
      --output-root {RESULTS}
elif not completed(CASE):
    print(f'SKIPPED optional case after {elapsed_hours:.2f}h to protect the ~8h budget')
show_case(CASE)

## 10. Final scoreboard and artifact archive

**Mục tiêu:** gom mọi fixed top-k và threshold policy vào một bảng xếp hạng để xem ngay dưới cell.  
**Đầu ra:** `khoan_diem_scoreboard.csv` cùng `drill_khoan_diem_results.tar.gz` chứa config, metrics, predictions, split IDs và provenance của các case hoàn tất.  
**Lưu ý:** bảng threshold đã được tune trên cùng dev split nên dùng để chọn policy cho lần đánh giá sau; không diễn giải phần tăng do threshold như một test-set gain độc lập.

In [ ]:
scoreboard = []
for metrics_path in sorted(RESULTS.glob('*/metrics_by_output_k.json')):
    payload = json.loads(metrics_path.read_text(encoding='utf-8'))
    for output_k, values in payload['results'].items():
        scoreboard.append({
            'experiment': metrics_path.parent.name,
            'policy': f'top_{output_k}',
            'precision': values['precision'],
            'recall': values['recall'],
            'f2': values['f2'],
        })
    threshold_path = metrics_path.parent / 'metrics_by_score_threshold.json'
    if threshold_path.exists():
        threshold_payload = json.loads(threshold_path.read_text(encoding='utf-8'))
        for fallback, values in threshold_payload['best_by_fallback_k'].items():
            scoreboard.append({
                'experiment': metrics_path.parent.name,
                'policy': f'threshold_fallback_{fallback}',
                'precision': values['precision'],
                'recall': values['recall'],
                'f2': values['f2'],
            })
scoreboard_df = pd.DataFrame(scoreboard).sort_values('f2', ascending=False)
display(scoreboard_df.reset_index(drop=True))
scoreboard_df.to_csv(RESULTS / 'khoan_diem_scoreboard.csv', index=False)

!tar -czf /kaggle/working/drill_khoan_diem_results.tar.gz -C /kaggle/working results
!ls -lh /kaggle/working/drill_khoan_diem_results.tar.gz